# TP3 - Procesamiento del Lenguaje Natural (NLP)
**Inteligencia Artificial y Neurociencias**

## 4. Large Language Models (LLMs)

En esta parte se evalúa un **LLM generativo ejecutado localmente, Qwen2.5-1.5B-Instruct** (`Qwen/Qwen2.5-1.5B-Instruct`), **sin ningún tipo de entrenamiento**: toda la adaptación a la tarea se hace mediante el *prompt*. Se exploran:

1. **Zero-shot:** distintas instrucciones (con y sin el título de la propuesta, con y sin definiciones de las clases, en español y en inglés).
2. **Few-shot:** con la mejor instrucción, se agregan ejemplos de train **aleatorios y balanceados por clase**, variando la cantidad de ejemplos (*k* = 1, 3 y 5 por clase) y repitiendo cada selección con 3 semillas para medir cuánto depende el resultado de qué ejemplos tocaron.
3. **Calibración:** como el método de predicción devuelve probabilidades, se prueba además una corrección del sesgo del modelo hacia ciertas etiquetas (*contextual calibration*), que no usa etiquetas ni entrenamiento.

**Cómo se obtiene la predicción (scoring de etiquetas).** En lugar de dejar que el modelo genere texto libre y parsearlo, se le pide que responda con una sola palabra (`favor`, `contra` o `neutral`) y se mira la **probabilidad que el modelo asigna a cada etiqueta como primer token de la respuesta**. La predicción es la etiqueta más probable. Ventajas: nunca hay respuestas inválidas, se necesita una sola pasada hacia adelante por comentario (no hay decodificación autoregresiva) y se obtienen probabilidades, que permiten calibrar.

**Protocolo de evaluación** (el mismo de las partes 1 a 3):
- Se usan exactamente los conjuntos de `splits_tp3.csv` (split agrupado por propuesta, ≈70/15/15). Los ejemplos few-shot se toman **solo de train**.
- Todas las decisiones (instrucción, cantidad de ejemplos, calibración) se toman con **validación**. El conjunto de **test se usa una única vez**, al final (sección 4.7).
- Métrica principal: **Macro-F1** respecto de `majority_label`. Se reportan además accuracy, F1 por clase, tiempo de inferencia, cantidad de tokens por prompt y uso de VRAM.

> **Para correr en Colab:** activar la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*) y subir `splits_tp3.csv` (generado por la parte 1). Opcionalmente, subir `resultados_parte2_test.csv` y `resultados_parte3_test.csv` para la comparación de la sección 4.8.4. El modelo no requiere token de Hugging Face. Todo el notebook tarda aproximadamente 20–30 minutos en una T4.

### 4.0. Configuración

In [ ]:
!pip -q install datasets

In [ ]:
import os
import gc
import time
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

import sklearn
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
CLASES = ["favor", "against", "neutral"]   # mismo orden que en las partes 1 a 3
MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

sns.set_theme(style="whitegrid")

def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fijar_semilla(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32   # la T4 no soporta bf16 de forma eficiente

def sincronizar():
    """Espera a que termine el trabajo en la GPU (necesario para medir tiempos correctamente)."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# Hardware / versiones (para reportar costos)
print("Python:", platform.python_version(), "| torch:", torch.__version__,
      "| transformers:", transformers.__version__, "| scikit-learn:", sklearn.__version__)
print("CPU:", platform.processor() or platform.machine(), "| núcleos:", os.cpu_count())
if DEVICE.type == "cuda":
    prop = torch.cuda.get_device_properties(0)
    print(f"GPU: {prop.name} | VRAM total: {prop.total_memory / 1e9:.1f} GB")
else:
    print("ATENCIÓN: no hay GPU disponible. La inferencia del LLM en CPU es muy lenta; "
          "activar la GPU en Colab antes de seguir.")

### 4.1. Carga de datos y particiones

Se reconstruye `df_modelado` igual que en las partes anteriores (se descartan los 150 comentarios sin mayoría) y se asigna a cada comentario su conjunto según `splits_tp3.csv`.

In [ ]:
dataset = load_dataset("SINAI/ALIA-es-discriminative-stance-detection")
df = dataset["train"].to_pandas()

cols_anotadores = ["annotation_1", "annotation_2", "annotation_3"]
df_modelado = df[df["majority_label"].isin(CLASES)].copy().reset_index(drop=True)
df_modelado["label"] = df_modelado["majority_label"]
df_modelado["id"] = df_modelado["id"].astype(str)

# Nivel de acuerdo entre anotadores (se usa en el análisis de errores)
df_modelado["max_acuerdo"] = df_modelado[cols_anotadores].apply(lambda f: f.value_counts().max(), axis=1)
df_modelado["nivel_acuerdo"] = df_modelado["max_acuerdo"].map({3: "Total (3/3)", 2: "Parcial (2/3)"})

RUTA_SPLITS = "splits_tp3.csv"
if not os.path.exists(RUTA_SPLITS):
    try:
        from google.colab import files
        print("No se encontró splits_tp3.csv: subilo (lo genera la parte 1).")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Falta splits_tp3.csv (lo genera la parte 1).")

splits = pd.read_csv(RUTA_SPLITS, dtype={"id": str})
df_modelado = df_modelado.merge(splits, on="id", how="left", validate="one_to_one")
assert df_modelado["split"].notna().all(), "Hay comentarios sin split asignado"

train_df = df_modelado[df_modelado["split"] == "train"].reset_index(drop=True)
val_df = df_modelado[df_modelado["split"] == "val"].reset_index(drop=True)
test_df = df_modelado[df_modelado["split"] == "test"].reset_index(drop=True)

# Controles: todos los comentarios tienen split y no hay propuestas compartidas
assert len(train_df) + len(val_df) + len(test_df) == len(df_modelado) == 2850
print(f"Train: {len(train_df)} | Validación: {len(val_df)} | Test: {len(test_df)}")
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert not set(a["target"]) & set(b["target"])

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

display(pd.crosstab(df_modelado["split"], df_modelado["label"]).loc[["train", "val", "test"], CLASES])

### 4.2. Carga del modelo

Qwen2.5-1.5B-Instruct es un modelo *decoder-only* de ≈1.500 M de parámetros (≈14 veces más que BETO), entrenado con instrucciones y multilingüe (incluye español). Se carga en fp16, lo que ocupa ≈3 GB de VRAM: entra cómodo en una T4 sin cuantizar.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_LLM)
tokenizer.padding_side = "right"
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

sincronizar(); t0 = time.perf_counter()
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, torch_dtype=DTYPE).to(DEVICE).eval()
sincronizar(); T_CARGA_S = time.perf_counter() - t0
assert next(llm.parameters()).dtype == DTYPE

N_PARAMS_M = sum(p.numel() for p in llm.parameters()) / 1e6
TAMAÑO_LLM_MB = sum(p.numel() * p.element_size() for p in llm.parameters()) / 1e6
VRAM_MODELO_GB = torch.cuda.memory_allocated() / 1e9 if DEVICE.type == "cuda" else np.nan
print(f"Parámetros: {N_PARAMS_M:.0f} M | tamaño en {str(DTYPE).replace('torch.', '')}: {TAMAÑO_LLM_MB:.0f} MB | "
      f"VRAM ocupada por los pesos: {VRAM_MODELO_GB:.2f} GB | carga: {T_CARGA_S:.1f} s")

### 4.3. Prompts y scoring de etiquetas

#### Instrucciones a comparar (zero-shot)

Todas las instrucciones van en el mensaje de sistema y el comentario en el mensaje de usuario, con el formato de chat del modelo (`apply_chat_template`). Se varían tres aspectos:

| Prompt | Incluye la propuesta | Definiciones de las clases | Idioma | Etiquetas |
|---|---|---|---|---|
| `basico_sin_propuesta` | No | No | Español | favor / contra / neutral |
| `basico` | Sí (título) | No | Español | favor / contra / neutral |
| `definiciones` | Sí (título) | Sí | Español | favor / contra / neutral |
| `definiciones_en` | Sí (título) | Sí | Inglés | favor / against / neutral |

- **Propuesta:** como en la parte 3, la postura se define respecto de una propuesta, así que se espera que mostrarla ayude. Se usa solo el título (la descripción es muy larga, ver parte 3).
- **Definiciones:** un modelo chico puede no tener claro qué cuenta como *neutral* (preguntas, información, comentarios ambiguos). Las definiciones explicitan el criterio.
- **Idioma:** los LLMs suelen seguir mejor las instrucciones en inglés (la mayor parte de su entrenamiento de instrucciones es en inglés), aunque los comentarios estén en español.

In [ ]:
DEF_ES = """Definiciones:
- favor: el comentario apoya la propuesta, la defiende o da argumentos para que se apruebe.
- contra: el comentario rechaza o critica la propuesta, o argumenta que es innecesaria, perjudicial o inviable.
- neutral: el comentario no toma una posición clara sobre la propuesta (por ejemplo, hace una pregunta, aporta información, habla de otro tema o es ambiguo)."""

DEF_EN = """Definitions:
- favor: the comment supports the proposal, defends it or gives arguments for it to be approved.
- against: the comment rejects or criticizes the proposal, or argues that it is unnecessary, harmful or unfeasible.
- neutral: the comment does not take a clear position on the proposal (for example, it asks a question, provides information, talks about something else or is ambiguous)."""

TAREA_ES = ("Tu tarea es clasificar la postura de comentarios escritos por ciudadanos sobre propuestas "
            "publicadas en una plataforma de participación ciudadana. Indicá si el comentario está "
            "a favor de la propuesta, en contra de la propuesta o es neutral.")
TAREA_EN = ("Your task is to classify the stance of comments written by citizens about proposals published "
            "on a citizen participation platform. The proposal and the comment are in Spanish. Indicate "
            "whether the comment is in favor of the proposal, against the proposal or neutral.")
RESP_ES = "Respondé con una sola palabra: favor, contra o neutral."
RESP_EN = "Answer with a single word: favor, against or neutral."

PROMPTS = {
    "basico_sin_propuesta": {
        "idioma": "es", "con_propuesta": False,
        "system": TAREA_ES + " Solo vas a ver el comentario, no la propuesta.\n" + RESP_ES},
    "basico": {
        "idioma": "es", "con_propuesta": True,
        "system": TAREA_ES + "\n" + RESP_ES},
    "definiciones": {
        "idioma": "es", "con_propuesta": True,
        "system": TAREA_ES + "\n\n" + DEF_ES + "\n\n" + RESP_ES},
    "definiciones_en": {
        "idioma": "en", "con_propuesta": True,
        "system": TAREA_EN + "\n\n" + DEF_EN + "\n\n" + RESP_EN},
}

# Palabra con la que el modelo debe responder para cada clase
VERBAL = {"es": {"favor": "favor", "against": "contra", "neutral": "neutral"},
          "en": {"favor": "favor", "against": "against", "neutral": "neutral"}}
ROTULOS = {"es": ("Propuesta", "Comentario", "Postura:"),
           "en": ("Proposal", "Comment", "Stance:")}

def mensaje_usuario(cfg, propuesta, comentario):
    rot_prop, rot_com, pregunta = ROTULOS[cfg["idioma"]]
    partes = [f'{rot_prop}: "{propuesta}"'] if cfg["con_propuesta"] else []
    partes += [f'{rot_com}: "{comentario}"', pregunta]
    return "\n".join(partes)

def armar_prompt(cfg, fila, demos=None):
    """Arma el prompt completo en formato chat. Los ejemplos few-shot (demos) van como turnos
    previos de usuario / asistente, con la respuesta correcta como mensaje del asistente."""
    msgs = [{"role": "system", "content": cfg["system"]}]
    if demos is not None:
        for _, d in demos.iterrows():
            msgs.append({"role": "user", "content": mensaje_usuario(cfg, d["target"], d["comment"])})
            msgs.append({"role": "assistant", "content": VERBAL[cfg["idioma"]][d["label"]]})
    msgs.append({"role": "user", "content": mensaje_usuario(cfg, fila["target"], fila["comment"])})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(armar_prompt(PROMPTS["definiciones"], val_df.iloc[0]))

#### Scoring de etiquetas

Para cada prompt se hace **una sola pasada hacia adelante** y se toma la distribución del modelo sobre el **siguiente token** (el primero de la respuesta del asistente). El puntaje de cada clase es la probabilidad de que la respuesta empiece con su etiqueta (sumando las variantes en minúscula y con mayúscula inicial, p. ej. `favor` / `Favor`). Las tres probabilidades se renormalizan entre sí y la predicción es la clase más probable.

Se registra además la **masa de probabilidad en las etiquetas**: qué fracción de la probabilidad total del modelo cae en alguno de esos tokens. Si es alta, el modelo "entendió" el formato de respuesta; si es baja, estaría respondiendo otra cosa y el scoring fuerza una de las tres clases.

Detalles de eficiencia: los prompts se ordenan por longitud y se agrupan en lotes con un presupuesto de tokens (*padding* mínimo), y la proyección al vocabulario se calcula solo en la última posición de cada prompt.

In [ ]:
def ids_primer_token(palabra):
    """Ids del primer token de la etiqueta, en minúscula y con mayúscula inicial."""
    return sorted({tokenizer.encode(v, add_special_tokens=False)[0] for v in (palabra, palabra.capitalize())})

IDS_ETIQUETA = {}
for idioma, verbal in VERBAL.items():
    IDS_ETIQUETA[idioma] = {c: ids_primer_token(verbal[c]) for c in CLASES}
    todos = [i for ids in IDS_ETIQUETA[idioma].values() for i in ids]
    assert len(todos) == len(set(todos)), f"Dos clases comparten el primer token ({idioma})"
    for c in CLASES:
        print(f"[{idioma}] {c:8s} → '{verbal[c]}' se tokeniza como "
              f"{[tokenizer.decode([t]) for t in tokenizer.encode(verbal[c], add_special_tokens=False)]} | "
              f"primeros tokens usados: {[tokenizer.decode([t]) for t in IDS_ETIQUETA[idioma][c]]}")

MAX_TOKENS_LOTE = 16384   # presupuesto de tokens por lote (bajarlo si hay errores de memoria)

@torch.no_grad()
def puntuar(prompts, idioma):
    """Devuelve (log-prob de cada clase [n, 3], masa de probabilidad en las etiquetas [n], tokens por prompt [n])."""
    ids_clase = IDS_ETIQUETA[idioma]
    ids_todos = [i for c in CLASES for i in ids_clase[c]]
    enc = [tokenizer(p, add_special_tokens=False)["input_ids"] for p in prompts]
    n_tok = np.array([len(e) for e in enc])
    orden = np.argsort(-n_tok)   # los más largos primero: si falta memoria, falla al principio

    decoder, lm_head = llm.get_decoder(), llm.get_output_embeddings()
    logp = np.zeros((len(prompts), len(CLASES)))
    masa = np.zeros(len(prompts))

    i = 0
    while i < len(orden):
        largo = n_tok[orden[i]]                          # el primero del lote es el más largo
        tam = max(1, min(64, MAX_TOKENS_LOTE // largo))
        idx = orden[i:i + tam]
        input_ids = torch.full((len(idx), largo), PAD_ID, dtype=torch.long)
        attn = torch.zeros((len(idx), largo), dtype=torch.long)
        for r, j in enumerate(idx):
            input_ids[r, :n_tok[j]] = torch.tensor(enc[j])
            attn[r, :n_tok[j]] = 1
        input_ids, attn = input_ids.to(DEVICE), attn.to(DEVICE)

        h = decoder(input_ids=input_ids, attention_mask=attn).last_hidden_state
        h_ult = h[torch.arange(len(idx), device=DEVICE), attn.sum(1) - 1]   # última posición real (padding a derecha)
        lp = torch.log_softmax(lm_head(h_ult).float(), dim=-1)
        for k, c in enumerate(CLASES):
            logp[idx, k] = torch.logsumexp(lp[:, ids_clase[c]], dim=-1).cpu().numpy()
        masa[idx] = lp[:, ids_todos].exp().sum(-1).cpu().numpy()
        i += tam

    if np.isnan(logp).any():
        print("ATENCIÓN: hay NaN en las log-probabilidades (posible overflow en fp16). "
              "Probar con DTYPE = torch.float32.")
    return logp, masa, n_tok

def softmax(x):
    e = np.exp(x - x.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)

def predecir_clases(logp, p_cf=None):
    """Predicción a partir de las log-probs. Si se pasa p_cf, aplica la calibración contextual."""
    p = softmax(logp)
    if p_cf is not None:
        p = p / p_cf
    return np.array(CLASES)[p.argmax(1)]

def metricas(y_true, y_pred):
    f1s = f1_score(y_true, y_pred, labels=CLASES, average=None, zero_division=0)
    return {"macro_f1": f1_score(y_true, y_pred, labels=CLASES, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred),
            **{f"f1_{c}": f for c, f in zip(CLASES, f1s)}}

COLS_METRICAS = ["macro_f1", "accuracy"] + [f"f1_{c}" for c in CLASES]
COLS_DIST = [f"%pred_{c}" for c in CLASES]

def distribucion(y_pred):
    return {f"%pred_{c}": (y_pred == c).mean() * 100 for c in CLASES}

#### Calibración contextual

Los LLMs suelen tener un **sesgo a priori** hacia algunas respuestas (por la redacción del prompt, por cuál etiqueta aparece primero en la instrucción o por cuál es la última de los ejemplos few-shot), lo que con clases desbalanceadas castiga mucho al Macro-F1. La *contextual calibration* (Zhao et al., 2021) estima ese sesgo pasándole al modelo el mismo prompt con entradas **sin contenido** (`"N/A"`, `""`, `"..."`) en lugar de la propuesta y el comentario, y divide las probabilidades de cada clase por las obtenidas para esas entradas vacías. No usa ninguna etiqueta ni entrenamiento, solo 3 pasadas extra por configuración.

Para cada configuración se registran las dos versiones (sin y con calibración) y la elección también se hace en validación.

In [ ]:
ENTRADAS_VACIAS = [{"target": v, "comment": v} for v in ["N/A", "", "..."]]

def correr_config(prompt, df_split, demos=None):
    """Puntúa todos los comentarios de df_split con un prompt (y ejemplos few-shot opcionales).
    Devuelve las predicciones sin y con calibración y las medidas de costo."""
    cfg = PROMPTS[prompt]
    prompts = [armar_prompt(cfg, fila, demos) for _, fila in df_split.iterrows()]
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sincronizar(); t0 = time.perf_counter()
    logp, masa, n_tok = puntuar(prompts, cfg["idioma"])
    logp_cf, _, _ = puntuar([armar_prompt(cfg, v, demos) for v in ENTRADAS_VACIAS], cfg["idioma"])
    sincronizar(); t_inf = time.perf_counter() - t0
    p_cf = softmax(logp_cf).mean(0)
    return {"pred": {False: predecir_clases(logp), True: predecir_clases(logp, p_cf)},
            "logp": logp, "p_cf": p_cf,
            "costo": {"tokens_prompt_medio": n_tok.mean(), "masa_etiquetas": masa.mean(),
                      "t_inf_s": t_inf, "t_inf_por_1000_s": t_inf / len(df_split) * 1000,
                      "vram_pico_GB": torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else np.nan}}

resultados = []   # registro de todos los experimentos sobre validación

def registrar(grupo, experimento, salida, y_true, **extra):
    for cal in [False, True]:
        y_pred = salida["pred"][cal]
        resultados.append({"grupo": grupo, "experimento": experimento, "calibracion": cal,
                           **metricas(y_true, y_pred), **distribucion(y_pred), **salida["costo"], **extra})

# Calentamiento (la primera llamada en GPU incluye inicializaciones que no son parte del costo real)
_ = correr_config("basico", val_df.head(16))

### 4.4. Zero-shot: comparación de instrucciones

In [ ]:
for nombre in PROMPTS:
    salida = correr_config(nombre, val_df)
    registrar("zero_shot", nombre, salida, y_val, prompt=nombre, k_por_clase=0, seed_demos=np.nan)
    print(f"{nombre:22s} | Macro-F1 val = {metricas(y_val, salida['pred'][False])['macro_f1']:.3f} "
          f"(calibrado: {metricas(y_val, salida['pred'][True])['macro_f1']:.3f}) | "
          f"{salida['costo']['t_inf_s']:.1f} s | p_cf = {dict(zip(CLASES, salida['p_cf'].round(2)))}")

zs = pd.DataFrame([r for r in resultados if r["grupo"] == "zero_shot"])
display(zs.set_index(["experimento", "calibracion"])[COLS_METRICAS + COLS_DIST +
        ["masa_etiquetas", "tokens_prompt_medio", "t_inf_por_1000_s"]].round(3))
print("Distribución real en validación (%):", (pd.Series(y_val).value_counts(normalize=True) * 100)
      .reindex(CLASES).round(1).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=zs.assign(calibracion=zs["calibracion"].map({False: "Sin calibrar", True: "Calibrado"})),
            x="experimento", y="macro_f1", hue="calibracion", ax=axes[0])
axes[0].axhline(0.26, ls=":", c="gray", label="Clasificador trivial (≈0,26)")
axes[0].set_title("Zero-shot: Macro-F1 en validación"); axes[0].set_xlabel(""); axes[0].set_ylabel("Macro-F1")
axes[0].tick_params(axis="x", rotation=15); axes[0].legend(fontsize=8)
for cont in axes[0].containers:
    axes[0].bar_label(cont, fmt="%.2f", fontsize=8)

# Distribución de las predicciones (sin calibrar) vs. la real
dist = zs[~zs["calibracion"]].set_index("experimento")[COLS_DIST]
dist.columns = CLASES
dist.loc["(real en validación)"] = (pd.Series(y_val).value_counts(normalize=True) * 100).reindex(CLASES).to_numpy()
dist.plot.barh(stacked=True, ax=axes[1], color=["C2", "C3", "C7"])
axes[1].set_xlabel("% de comentarios"); axes[1].set_ylabel("")
axes[1].set_title("Distribución de predicciones (sin calibrar)"); axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

MEJOR_ZS = zs.loc[zs["macro_f1"].idxmax()]
PROMPT = MEJOR_ZS["experimento"]
print(f"Mejor zero-shot: prompt '{PROMPT}' | calibración = {MEJOR_ZS['calibracion']} "
      f"→ Macro-F1 (val) = {MEJOR_ZS['macro_f1']:.3f}")
print(f"Se usa el prompt '{PROMPT}' para los experimentos few-shot.")

**Observaciones** *(completar con los resultados)*

- **Propuesta:** ¿mostrar el título de la propuesta mejora respecto de `basico_sin_propuesta`? A diferencia de la bolsa de palabras de la parte 2, un LLM puede relacionar comentario y propuesta en lenguaje natural.
- **Definiciones e idioma:** ¿explicitar qué es *neutral* ayuda a un modelo de 1,5B? ¿La instrucción en inglés funciona mejor o peor?
- **Sesgo hacia una clase:** mirar la distribución de las predicciones. Es habitual que un modelo chico casi nunca prediga una de las clases (típicamente *neutral*, o al revés, que la use como "comodín"), lo que hunde su F1 y el Macro-F1. La masa de probabilidad en las etiquetas indica si el modelo respeta el formato de respuesta.
- **Calibración:** ¿corregir el sesgo con entradas vacías mejora el Macro-F1? Si la distribución calibrada se acerca a la real, el problema del zero-shot era en buena parte de sesgo y no de "comprensión".

### 4.5. Few-shot: cantidad de ejemplos y variabilidad de la selección

Con el mejor prompt de 4.4 se agregan *k* ejemplos **por clase** (*k* = 1, 3, 5, es decir 3, 9 y 15 ejemplos en total), elegidos **al azar y balanceados** de train y presentados en orden aleatorio como turnos previos de la conversación. Los mismos ejemplos se usan para todos los comentarios de una corrida.

Como el resultado puede depender mucho de *qué* ejemplos tocaron (y de su orden), cada *k* se repite con **3 semillas** de selección. Se reportan la media y el desvío.

Los ejemplos se toman de propuestas de train, que no aparecen en validación ni test (split agrupado), así que no hay filtración.

In [ ]:
K_POR_CLASE = [1, 3, 5]
SEEDS_DEMOS = [0, 1, 2]

def elegir_demos(k, seed):
    """k ejemplos al azar por clase, de train, en orden aleatorio."""
    rng = np.random.RandomState(seed)
    demos = pd.concat([train_df[train_df["label"] == c].sample(k, random_state=rng) for c in CLASES])
    return demos.sample(frac=1, random_state=rng)

print("Ejemplo de prompt few-shot (k=1 por clase):\n")
print(armar_prompt(PROMPTS[PROMPT], val_df.iloc[0], elegir_demos(1, 0)))

In [ ]:
for k in K_POR_CLASE:
    for seed in SEEDS_DEMOS:
        demos = elegir_demos(k, seed)
        salida = correr_config(PROMPT, val_df, demos)
        registrar("few_shot", f"k={k} | seed={seed}", salida, y_val,
                  prompt=PROMPT, k_por_clase=k, seed_demos=seed)
        print(f"k={k} por clase | seed={seed} | Macro-F1 val = "
              f"{metricas(y_val, salida['pred'][False])['macro_f1']:.3f} "
              f"(calibrado: {metricas(y_val, salida['pred'][True])['macro_f1']:.3f}) | "
              f"{salida['costo']['tokens_prompt_medio']:.0f} tokens/prompt | {salida['costo']['t_inf_s']:.1f} s | "
              f"VRAM pico {salida['costo']['vram_pico_GB']:.2f} GB")

fs = pd.DataFrame([r for r in resultados if r["grupo"] == "few_shot"])
display(fs.set_index(["experimento", "calibracion"])[COLS_METRICAS + COLS_DIST +
        ["masa_etiquetas", "tokens_prompt_medio", "t_inf_por_1000_s", "vram_pico_GB"]].round(3))

In [ ]:
# Resumen por k (media y desvío entre semillas); k=0 es el zero-shot con el mismo prompt
zs_prompt = zs[zs["experimento"] == PROMPT].assign(k_por_clase=0)
por_k = (pd.concat([zs_prompt, fs])
         .groupby(["k_por_clase", "calibracion"])
         .agg(macro_f1_media=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
              **{f"{c}_media": (c, "mean") for c in [f"f1_{x}" for x in CLASES]},
              tokens_prompt_medio=("tokens_prompt_medio", "mean"),
              t_inf_por_1000_s=("t_inf_por_1000_s", "mean"), vram_pico_GB=("vram_pico_GB", "max"))
         .reset_index())
display(por_k.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for cal, estilo in [(False, "o-"), (True, "s--")]:
    g = por_k[por_k["calibracion"] == cal]
    axes[0].errorbar(g["k_por_clase"], g["macro_f1_media"], yerr=g["macro_f1_std"].fillna(0), fmt=estilo,
                     capsize=4, label="Calibrado" if cal else "Sin calibrar")
    # puntos de cada semilla
    s = fs[fs["calibracion"] == cal]
    axes[0].scatter(s["k_por_clase"], s["macro_f1"], alpha=0.35, s=15, color="C1" if cal else "C0")
axes[0].set_xlabel("Ejemplos por clase (k)"); axes[0].set_ylabel("Macro-F1 (validación)")
axes[0].set_title("Macro-F1 vs. cantidad de ejemplos"); axes[0].set_xticks([0] + K_POR_CLASE); axes[0].legend()

g = por_k[~por_k["calibracion"]]
for c in CLASES:
    axes[1].plot(g["k_por_clase"], g[f"f1_{c}_media"], marker="o", label=c)
axes[1].set_xlabel("Ejemplos por clase (k)"); axes[1].set_ylabel("F1 (validación, sin calibrar)")
axes[1].set_title("F1 por clase vs. k"); axes[1].set_xticks([0] + K_POR_CLASE); axes[1].legend()

axes[2].plot(g["k_por_clase"], g["tokens_prompt_medio"], marker="o", c="C4")
axes[2].set_xlabel("Ejemplos por clase (k)"); axes[2].set_ylabel("Tokens por prompt (media)", color="C4")
ax2 = axes[2].twinx()
ax2.plot(g["k_por_clase"], g["t_inf_por_1000_s"], marker="s", c="C5")
ax2.set_ylabel("Segundos por 1.000 comentarios", color="C5"); ax2.grid(False)
axes[2].set_title("Costo vs. k"); axes[2].set_xticks([0] + K_POR_CLASE)
plt.tight_layout(); plt.show()

# Elección de k y calibración por la media de las 3 semillas
fila_k = por_k[por_k["k_por_clase"] > 0].sort_values("macro_f1_media", ascending=False).iloc[0]
K_FINAL, CAL_FS = int(fila_k["k_por_clase"]), bool(fila_k["calibracion"])
print(f"Mejor few-shot: k={K_FINAL} por clase | calibración = {CAL_FS} → "
      f"Macro-F1 (val) = {fila_k['macro_f1_media']:.3f} ± {fila_k['macro_f1_std']:.3f}")

**Observaciones** *(completar con los resultados)*

- ¿Agregar ejemplos mejora respecto del zero-shot? ¿La mejora sigue creciendo con *k* o se satura? Un modelo de 1,5B tiene una capacidad limitada de aprovechar contexto largo; con 15 ejemplos el prompt ya tiene ≈**…** tokens.
- **Variabilidad:** ¿qué tan grande es el desvío entre semillas comparado con la diferencia entre valores de *k*? Si es del mismo orden, el few-shot es frágil: el resultado depende de qué ejemplos tocaron y en qué orden.
- **Por clase:** ¿qué clase se beneficia más de los ejemplos? Es esperable que *neutral*, la más difícil de definir con palabras, sea la que más gana al ver casos concretos.
- **Calibración:** ¿sigue ayudando con ejemplos? Los ejemplos también introducen sesgos (por ejemplo, hacia la etiqueta del último ejemplo), que la calibración puede compensar.
- **Costo:** el tiempo de inferencia crece aproximadamente lineal con la cantidad de tokens del prompt. Pasar de zero-shot a *k*=5 multiplica el costo por ≈**…**.

### 4.6. Resumen de los experimentos en validación

In [ ]:
res_val = pd.DataFrame(resultados)
res_val["nombre"] = (res_val["grupo"] + " | " + res_val["experimento"] +
                     res_val["calibracion"].map({False: "", True: " | calibrado"}))

fig, ax = plt.subplots(figsize=(9, 0.28 * len(res_val) + 1))
sns.barplot(data=res_val.sort_values(["grupo", "k_por_clase", "experimento", "calibracion"], ascending=[False, True, True, True]),
            y="nombre", x="macro_f1", hue="grupo", dodge=False, ax=ax)
ax.axvline(0.26, ls=":", c="gray")
ax.set_xlabel("Macro-F1 (validación)"); ax.set_ylabel(""); ax.tick_params(axis="y", labelsize=7)
ax.set_title("Experimentos de la parte 4 (Qwen2.5-1.5B-Instruct)"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

CONFIG_ZS = {"prompt": PROMPT, "calibracion": bool(MEJOR_ZS["calibracion"])}
CONFIG_FS = {"prompt": PROMPT, "k_por_clase": K_FINAL, "calibracion": CAL_FS}
print("Configuración final zero-shot:", CONFIG_ZS)
print("Configuración final few-shot:", CONFIG_FS, "| semillas de selección:", SEEDS_DEMOS)

### 4.7. Evaluación final en test

Recién ahora se usa el conjunto de test, una única vez, para:
1. **Zero-shot:** el mejor prompt (y la decisión de calibrar) elegidos en 4.4.
2. **Few-shot:** el mejor *k* (y la decisión de calibrar) elegidos en 4.5, con las **mismas 3 selecciones de ejemplos** (semillas 0, 1 y 2). Se reportan la media y el desvío.

No hay entrenamiento, así que el costo es solo de inferencia (más la carga del modelo, que se hace una vez). El tiempo de inferencia incluye las 3 pasadas extra de la calibración.

In [ ]:
res_test, preds_test = [], {}

def evaluar_en_test(nombre, prompt, calibracion, demos=None, **extra):
    salida = correr_config(prompt, test_df, demos)
    y_pred = salida["pred"][calibracion]
    preds_test[nombre] = y_pred
    res_test.append({"modelo": nombre, **metricas(y_test, y_pred), **distribucion(y_pred),
                     "t_train_s": 0.0, **salida["costo"], "tamaño_modelo_MB": TAMAÑO_LLM_MB, **extra})
    print(nombre, f"→ Macro-F1 test = {res_test[-1]['macro_f1']:.3f} | {salida['costo']['t_inf_s']:.1f} s")

NOMBRE_ZS = f"Qwen2.5-1.5B zero-shot ({PROMPT}{', calibrado' if CONFIG_ZS['calibracion'] else ''})"
evaluar_en_test(NOMBRE_ZS, PROMPT, CONFIG_ZS["calibracion"], seed_demos=np.nan)

for seed in SEEDS_DEMOS:
    evaluar_en_test(f"Qwen2.5-1.5B few-shot k={K_FINAL} (seed={seed})", PROMPT, CAL_FS,
                    elegir_demos(K_FINAL, seed), seed_demos=seed)

In [ ]:
test_tabla = pd.DataFrame(res_test).set_index("modelo")
fs_test = test_tabla[test_tabla.index.str.contains("few-shot")]
fila_media, fila_std = fs_test.mean(numeric_only=True), fs_test.std(numeric_only=True)
NOMBRE_FS_MEDIA = f"Qwen2.5-1.5B few-shot k={K_FINAL} (media de 3 selecciones)"
NOMBRE_FS_STD = f"Qwen2.5-1.5B few-shot k={K_FINAL} (desvío entre selecciones)"
test_tabla.loc[NOMBRE_FS_MEDIA] = fila_media
test_tabla.loc[NOMBRE_FS_STD] = fila_std
test_tabla.loc[[NOMBRE_FS_MEDIA, NOMBRE_FS_STD], "seed_demos"] = np.nan

display(test_tabla.drop(columns="seed_demos")[COLS_METRICAS + COLS_DIST +
        ["tokens_prompt_medio", "masa_etiquetas", "t_inf_s", "t_inf_por_1000_s", "vram_pico_GB", "tamaño_modelo_MB"]].round(3))
print(f"Macro-F1 test few-shot: {fila_media['macro_f1']:.3f} ± {fila_std['macro_f1']:.3f} | "
      f"zero-shot: {test_tabla.loc[NOMBRE_ZS, 'macro_f1']:.3f}")

**Observaciones** *(completar con los resultados)*

- ¿El Macro-F1 de test es similar al de validación? Con un modelo sin entrenamiento el riesgo de sobreajuste es menor, pero la elección del prompt y de *k* sí se hizo mirando validación.
- ¿Cuánto mejora el few-shot respecto del zero-shot en test, y cuánto es el desvío entre selecciones de ejemplos?
- **Costos:** no hay entrenamiento, pero la inferencia es mucho más cara que la de BETO (más parámetros y prompts más largos): comparar segundos por 1.000 comentarios con las partes 2 y 3.

### 4.8. Análisis de resultados

#### 4.8.1. Matrices de confusión y F1 por clase

Para el few-shot se muestra la selección de ejemplos con Macro-F1 de validación más cercano a la mediana (la elección no usa el test).

In [ ]:
fs_val_final = fs[(fs["k_por_clase"] == K_FINAL) & (fs["calibracion"] == CAL_FS)]
seed_rep = int(fs_val_final.iloc[(fs_val_final["macro_f1"] - fs_val_final["macro_f1"].median()).abs().argmin()]["seed_demos"])
NOMBRE_FS = f"Qwen2.5-1.5B few-shot k={K_FINAL} (seed={seed_rep})"
print("Selección de ejemplos representativa: seed", seed_rep)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (titulo, nombre) in zip(axes, [("Zero-shot", NOMBRE_ZS), (f"Few-shot k={K_FINAL}", NOMBRE_FS)]):
    y_pred = preds_test[nombre]
    cm = confusion_matrix(y_test, y_pred, labels=CLASES, normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{titulo} — Macro-F1 = {f1_score(y_test, y_pred, average='macro'):.3f}")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real (majority_label)")
plt.suptitle("Matrices de confusión en test (normalizadas por fila)"); plt.tight_layout(); plt.show()

print(f"=== {NOMBRE_FS} ===")
print(classification_report(y_test, preds_test[NOMBRE_FS], labels=CLASES, digits=3))

cols_f1 = [f"f1_{c}" for c in CLASES] + ["macro_f1"]
x = np.arange(len(cols_f1))
fig, ax = plt.subplots(figsize=(8, 4))
b_zs = ax.bar(x - 0.2, test_tabla.loc[NOMBRE_ZS, cols_f1].to_numpy(dtype=float), 0.4, label="Zero-shot")
b_fs = ax.bar(x + 0.2, fila_media[cols_f1].to_numpy(dtype=float), 0.4,
              yerr=fila_std[cols_f1].to_numpy(dtype=float), capsize=4,
              label=f"Few-shot k={K_FINAL} (media ± desvío, 3 selecciones)")
ax.bar_label(b_zs, fmt="%.2f", fontsize=8); ax.bar_label(b_fs, fmt="%.2f", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(cols_f1)
ax.set_ylabel("F1 (test)"); ax.set_title("F1 por clase en test"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()

#### 4.8.2. Desempeño según el acuerdo entre anotadores

Si el LLM falla sobre todo en los comentarios donde los anotadores tampoco coincidieron, parte del error se explica por la ambigüedad de las etiquetas (techo de desempeño discutido en la parte 1).

In [ ]:
filas = []
for nombre in [NOMBRE_ZS, NOMBRE_FS]:
    for nivel in ["Total (3/3)", "Parcial (2/3)"]:
        m = (test_df["nivel_acuerdo"] == nivel).to_numpy()
        filas.append({"modelo": nombre, "nivel_acuerdo": nivel, "n": m.sum(),
                      "accuracy": accuracy_score(y_test[m], preds_test[nombre][m]),
                      "macro_f1": f1_score(y_test[m], preds_test[nombre][m], labels=CLASES,
                                           average="macro", zero_division=0)})
display(pd.DataFrame(filas).set_index(["modelo", "nivel_acuerdo"]).round(3))

#### 4.8.3. Ejemplos de errores del few-shot

In [ ]:
errores = test_df.assign(pred=preds_test[NOMBRE_FS])[lambda d: d["pred"] != d["label"]]
print(f"Errores en test: {len(errores)} de {len(test_df)}\n")
for (real, pred), g in errores.groupby(["label", "pred"]):
    if len(g) < 5:
        continue
    print(f"=== Real: {real} | Predicho: {pred} ({len(g)} casos) ===")
    for _, fila in g.sample(min(2, len(g)), random_state=SEED).iterrows():
        print(f"• [{fila['target'][:60]}] {fila['comment'][:250]}")
        print(f"  Anotaciones: {fila[cols_anotadores].tolist()}\n")

#### 4.8.4. Comparación con las partes 2 y 3

Si están disponibles `resultados_parte2_test.csv` y `resultados_parte3_test.csv`, se comparan desempeño y costo de todos los enfoques. La comparación completa se hace en la parte 5.

In [ ]:
bloques = []
COLS_COMP = ["macro_f1", "accuracy", "t_train_s", "t_inf_por_1000_s", "tamaño_modelo_MB"]
if os.path.exists("resultados_parte2_test.csv"):
    p2 = pd.read_csv("resultados_parte2_test.csv", index_col=0)
    p2["t_inf_por_1000_s"] = p2["t_inf_por_1000_ms"] / 1000
    bloques.append(p2[COLS_COMP].assign(enfoque="Clásico (parte 2)"))
if os.path.exists("resultados_parte3_test.csv"):
    p3 = pd.read_csv("resultados_parte3_test.csv", index_col=0)
    p3 = p3[p3.index.str.startswith("BETO sin fine-tuning") | (p3.index == "BETO fine-tuning (media de 3 semillas)")]
    bloques.append(p3[COLS_COMP].assign(enfoque="Encoder (parte 3)"))
bloques.append(test_tabla.loc[[NOMBRE_ZS, NOMBRE_FS_MEDIA], COLS_COMP].assign(enfoque="LLM (parte 4)"))

comp = pd.concat(bloques)
display(comp.round(4))

if len(bloques) > 1:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    marcadores = {"Clásico (parte 2)": "o", "Encoder (parte 3)": "s", "LLM (parte 4)": "D"}
    for i, (nombre, fila) in enumerate(comp.iterrows()):
        ax.scatter(fila["t_inf_por_1000_s"], fila["macro_f1"], s=80, color=f"C{i}",
                   marker=marcadores[fila["enfoque"]], label=nombre[:60])
    ax.set_xscale("log"); ax.set_xlabel("Tiempo de inferencia por 1.000 comentarios (s, escala log)")
    ax.set_ylabel("Macro-F1 (test)"); ax.set_title("Desempeño vs. costo de inferencia")
    ax.legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()
    print("Nota: los tiempos de la parte 2 se midieron en CPU y los de las partes 3 y 4 en GPU.")
else:
    print("No se encontraron los resultados de las partes 2 y 3: subirlos para ver la comparación (opcional).")

### 4.9. Guardado de resultados

Se guardan los resultados para la comparación entre enfoques (parte 5):
- `resultados_parte4_val.csv`: todos los experimentos (zero-shot y few-shot, con y sin calibración) sobre validación.
- `resultados_parte4_test.csv`: métricas, tiempos, tokens por prompt, VRAM y tamaño en test.
- `predicciones_parte4_test.csv`: predicciones en test de cada configuración (para comparaciones por comentario entre enfoques).

In [ ]:
res_val.drop(columns="nombre").to_csv("resultados_parte4_val.csv", index=False)

test_tabla.assign(modelo_llm=MODELO_LLM, prompt=PROMPT, calibracion_zs=CONFIG_ZS["calibracion"],
                  k_por_clase_fs=K_FINAL, calibracion_fs=CAL_FS, t_carga_modelo_s=T_CARGA_S,
                  vram_pesos_GB=VRAM_MODELO_GB,
                  gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
                  ).to_csv("resultados_parte4_test.csv")

pd.DataFrame({"id": test_df["id"], "label": y_test,
              "pred_zero_shot": preds_test[NOMBRE_ZS],
              **{f"pred_few_shot_k{K_FINAL}_seed{s}": preds_test[f"Qwen2.5-1.5B few-shot k={K_FINAL} (seed={s})"]
                 for s in SEEDS_DEMOS}}
             ).to_csv("predicciones_parte4_test.csv", index=False)
print("Archivos guardados.")

try:   # en Colab, descarga los archivos para no perderlos al cerrar el entorno
    from google.colab import files
    for f in ["resultados_parte4_val.csv", "resultados_parte4_test.csv", "predicciones_parte4_test.csv"]:
        files.download(f)
except ImportError:
    pass

### 4.10. Conclusiones de la parte 4 *(completar con los resultados)*

- **Zero-shot:** el mejor prompt fue **…** con un Macro-F1 en test de **X**. Mostrar la propuesta **(ayudó / no ayudó)**, las definiciones **(ayudaron / no)** y la instrucción en inglés **(funcionó mejor / peor)**. Sin calibrar, el modelo **(casi no predice / sobrepredice)** la clase **…**; la calibración contextual **(corrigió / no corrigió)** ese sesgo (**+…** puntos).
- **Few-shot:** con *k* = **…** ejemplos por clase el Macro-F1 en test es **Y ± s** (3 selecciones de ejemplos), **…** puntos **(más / menos)** que el zero-shot. Más ejemplos **(siguen mejorando / se saturan)**, y el desvío entre selecciones (**± …**) es **(chico / comparable)** respecto de esas diferencias.
- **Comparación:** sin ningún entrenamiento, Qwen2.5-1.5B queda **(por debajo / a la par / por encima)** del modelo clásico de la parte 2 (**…**) y **…** puntos por debajo de BETO con fine-tuning (**…**).
- **Costos:** no requiere entrenamiento ni datos etiquetados (salvo los pocos ejemplos del few-shot y la validación para elegir el prompt), pero el modelo ocupa ≈3 GB en fp16 y la inferencia cuesta ≈**…** s cada 1.000 comentarios en una T4 (≈**…** veces más que BETO), creciendo con la longitud del prompt.
- **Limitaciones:** se evaluó un único LLM chico; modelos más grandes (7B o más, o por API) probablemente rindan bastante mejor. El resultado del few-shot depende de qué ejemplos se eligen, y el scoring por primer token asume que el modelo responde con la etiqueta directamente (la masa de probabilidad en las etiquetas, ≈**…**, indica si esto se cumple).